In [69]:
import gurobipy as gp
from gurobipy import GRB

# Create a new model named "Anjos_Chapter2"
m_nt = gp.Model("Anjos_Chapter3-Network")

# Sets 
generators = [1, 2, 3]
time_periods = [1, 2, 3, 4, 5, 6]
nodes = [1, 2, 3, 4, 5, 6]

# lines
lines = [
    (1, 4), (1, 5), (1, 6),
    (2, 4), (2, 5), (2, 6),
    (3, 5), (3, 6)
]

# Parameters 
# Startup Costs (cU)
startup_cost = {1: 800, 2: 500, 3: 250}

# Variable Production Costs (c)
var_cost = {1: 5, 2: 15, 3: 30}

# Min and Max Power Output (P_min, P_max)
p_min_limit = {1: 80, 2: 50, 3: 30}
p_max_limit = {1: 300, 2: 200, 3: 100} 


# Startup and down limit (SU, SD)

start_up = {1: 100, 2: 70, 3: 40}
start_down = {1: 80, 2: 50, 3: 30}

# Ramp Up and Down Limits (RU, RD)
ramp_up = {1: 50, 2: 60, 3: 70} #RU1 from 50 to 60 
ramp_down = {1: 30, 2: 40, 3: 50} #RD1 from 30 to 50

#Timeup and Timedown 
time_up = {1: 3, 2: 2, 3: 1} #time unit has to stay on until can be allowed to shutdown
time_down = {1: 2, 2: 2, 3: 2}

# Demand (D) and Reserve (R)
#demand = {1: 240, 2: 250, 3: 200, 4: 170, 5: 260, 6: 190} #replace t=5 for 260 for 2.7.2 original D(1) = 240

# in form (n, t)
nodal_demand = {
    # Node 1 
    (1, 1): 0, (1, 2): 0, (1, 3): 0, (1, 4): 0, (1, 5): 0, (1, 6): 0,
    # Node 2 
    (2, 1): 0, (2, 2): 0, (2, 3): 0, (2, 4): 0, (2, 5): 0, (2, 6): 0,
    # Node 3 
    (3, 1): 0, (3, 2): 0, (3, 3): 0, (3, 4): 0, (3, 5): 0, (3, 6): 0,
    
    # Node 4
    (4, 1): 100, (4, 2): 100, (4, 3): 80,  (4, 4): 140, (4, 5): 100, (4, 6): 80,
    
    # Node 5 
    (5, 1): 90,  (5, 2): 100, (5, 3): 80,  (5, 4): 30,  (5, 5): 90,  (5, 6): 60,
    
    # Node 6 
    (6, 1): 50,  (6, 2): 50,  (6, 3): 40,  (6, 4): 0,   (6, 5): 40,  (6, 6): 50,
}

reserve = {t: 10 for t in time_periods}


# v[j,t]: 1 if unit j is on at time t, 0 otherwise
v = m_nt.addVars(generators, time_periods, vtype=GRB.BINARY, name="v")

# y[j,t]: 1 if unit j starts up at time t
y = m_nt.addVars(generators, time_periods, vtype=GRB.BINARY, name="y")

# z[j,t]: 1 if unit j shuts down at time t
z = m_nt.addVars(generators, time_periods, vtype=GRB.BINARY, name="z")

# p[j,t]: Power generated (Continuous, must be >= 0)
p = m_nt.addVars(generators, time_periods, lb=0, vtype=GRB.CONTINUOUS, name="p")

# Initial Conditions (State at t=0)

# 1. Initial Status (v_init): 1 = ON, 0 = OFF
# Unit 1 was ON, Units 2 & 3 were OFF
v_init = {1: 1, 2: 0, 3: 0}

# 2. Initial Power Output (p_init): in MW
# Unit 1 was producing 120 MW
p_init = {1: 120, 2: 0, 3: 0}

# 3. Initial Up-Time Counter (u_init): How many hours has the unit ALREADY been ON?
# Unit 1 has been on for 2 hours already. (Needed for Min Up Time constraint)
u_init = {1: 2, 2: 0, 3: 0}

# 4. Initial Down-Time Counter (d_init): How many hours has the unit ALREADY been OFF?

d_init = {1: 0, 2: 0, 3: 0}

#added netwrok data

#network capacity

#B_nm
B = {(n,m): 0.8 for (n,m) in lines}

for (n, m) in lines: 
    B[(m, n)] = 0.8

#P_nm - Transmission capacity of the line nm
P_line = {(n, m): 100 for (n, m) in lines}

for (n, m) in lines:
    P_line[(m, n)] = 100

#locating generating nodes
gen_node = {1: 1, 2: 2, 3: 3}

# Voltage Angle Variables
theta = m_nt.addVars(nodes, time_periods, lb=-GRB.INFINITY, vtype=GRB.CONTINUOUS, name="theta")

In [70]:
# Objective Function (2.1)

obj_func = gp.quicksum(
    var_cost[j] * p[j, t] + startup_cost[j] * y[j, t] 
    for j in generators 
    for t in time_periods)

m_nt.setObjective(obj_func, GRB.MINIMIZE)

 Our Objective function : $\min_{p_j(t)}\sum_{t \in T}\sum_{j \in J}{(c_j(t)p_j(t) + c_j^U(t)y_j(t)}) \qquad$                    (2.1)

In [71]:
#constraint 3.1 - replaced (2.2)

nodal_balance_constraints = m_nt.addConstrs(
    (
        gp.quicksum(p[j, t] for j in generators if gen_node[j] == n)
        - 
        nodal_demand[(n, t)]
        ==
        gp.quicksum(B.get((n, m), 0) * (theta[n, t] - theta[m, t]) for m in nodes if m != n)
        for n in nodes
        for t in time_periods
    ),
    name="Nodal_Balance"
)
                


Constraint 3.1

$$
\sum_{j \in \Lambda_n^G} p_j(t)  - \sum_{i \in \Lambda_n^D} D_i(t) = \sum_{m\in \Lambda_n^L} B_{nm}(\theta_n(t) - \theta_m(t))\qquad \forall n \in N, \forall t \in T
$$

Here $D_i(t)$ is the load of demand i in period t. $\Lambda_n^G$- set of generating units located at node n, $\Lambda_n^D$ - set of demands located at node n, and $\Lambda_n^L$ - the set of nodes directly connected to node n. $B_{nm}$ is a constant relating n and m.

In [72]:
#Constraint 3.2- Transimission capacity 

m_nt.addConstrs(
    (
        -P_line[(n,m)] 
        <= B[(n,m)] * (theta[n, t] - theta[m, t])
        for (n, m) in lines
        for t in time_periods
    ),
    name = f'Transmission_capacity_lower'
 )

m_nt.addConstrs(
    (
        B[(n,m)] * (theta[n, t] - theta[m, t])
        <= P_line[(n, m)]
        for (n, m) in lines
        for t in time_periods
    ),
    name=f'Transmission_capacity_upper'
)


{(1, 4, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 4): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 5): <gurobi.Constr *Awaiting Model Update*>,
 (1, 4, 6): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 4): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 5): <gurobi.Constr *Awaiting Model Update*>,
 (1, 5, 6): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 1): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 2): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 3): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 4): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 5): <gurobi.Constr *Awaiting Model Update*>,
 (1, 6, 6): <gurobi.Constr *Awaiting Model Update*>,
 (2, 4, 1): <gurobi.Constr *Awaiting Model Upd

Constraint 3.2- Transmission capacity

$$
-\overline{P}_{nm} \leq B_{nm}(\theta_n(t)-\theta_m(t)) \leq \overline{P}_{nm} \qquad \forall n \in N, \forall m \in \Lambda_n^L, \forall t \in T
$$

where, $\overline{P}_{nm}$ is the transmission capacity of line $nm$.



In [73]:
#Constraint 3.3- Reference angle

# Here we choose n=1 to be the reference node

m_nt.addConstrs(
    (
        theta[1, t] == 0 for t in time_periods
    ),
    name='reference_angle'
)

{1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>}

Constraint 3.3 - Reference angle

$$
\theta_{\hat{n}}(t) = 0 \qquad \forall t \in T
$$

where $\hat{n}$ is the node of the reference angle.

In [74]:
#adding maximum avaiable power- 2.3

p_max = m_nt.addVars(generators, time_periods, lb=0, vtype=GRB.CONTINUOUS, name="p_max")

#part of Constraint 2.4?
#for t in time_periods:
    #for j in generators:
        #m.addConstr(p_max[j, t] <= p_max_limit[j] * v[j, t],
                    #name= f"max_power_limit_{j}_{t}")

#Constraint 2.3 ### DIDNT THIS WORK? - I think it did, but I added the above constraint 
# to make sure that the max power is limited by the on/off status of the generator.

#for t in time_periods:
    #m.addConstr(
        #gp.quicksum(p_max[j, t] 
            #for j in generators
            #) >= demand[t] + reserve[t],
            #name = f"max_power_reserve_{j}_{t}")

#As we assume the area is the whole system

m_nt.addConstrs(
    (
        gp.quicksum(p_max[j, t] for j in generators) 
        >= gp.quicksum(nodal_demand[n, t] for n in nodes)  + reserve[t]
        for t in time_periods
    ),
    name=f"Max_power_constr"
)


{1: <gurobi.Constr *Awaiting Model Update*>,
 2: <gurobi.Constr *Awaiting Model Update*>,
 3: <gurobi.Constr *Awaiting Model Update*>,
 4: <gurobi.Constr *Awaiting Model Update*>,
 5: <gurobi.Constr *Awaiting Model Update*>,
 6: <gurobi.Constr *Awaiting Model Update*>}

Constr 3.4

$$
\sum_{j \in \Omega_r^G} \bar{p}_j(t) \geq \sum_{i \in \Omega_r^D}D_i(t) + R_i(t) \qquad \forall r \in \mathcal{R}, \forall t \in T
$$

where $\mathcal{R}$ - set of reserve areas. $R_i(t)$ reserve required in reserve area r in time period t. $\Omega_r^G$ - The set of generating units in area r. $\Omega_r^D$ - The set of demands in area r.

In [75]:
#Constraint 2.4  

#combined with constraint 2.11

$$
\bar{p}_j(t) \leq \bar{P}_j \times v[j, t] \qquad \forall t \in T, \forall j \in J
$$

In [76]:
#COnstraint 2.5

for t in time_periods:
    for j in generators:
        v_prev = v_init[j] if t == 1 else v[j, t-1]
        m_nt.addConstr(v_prev - v[j, t] + y[j, t] - z[j, t] == 0,
                    name = f"logic_binaries_{j}_{t}")

Constraint 2.5

$$
v_j(t-1) - v_j(t) + y_j(t) - z_j(t) = 0 \qquad \forall j \in J, \forall t \in T
$$

In [77]:
#Constraint 2.6 Starting-up Cost

#for t in time_periods:
    #for j in generators:
        #m.addConstr(
        #startup_cost >= 

$$
c_j^U(t) \geq \mathbf{k}_j^{\tau}\left(v_j(t) - \sum_{n=1}^{\tau}v_j(t-n) \right) \qquad \tau = 0,1,...,\tau_j^{cold} \quad \forall j \in J \quad \forall t \in T
$$

In [78]:
#Constraint 2.7 - Ramping-up constraints

for j in generators:
    for t in time_periods:
        p_prev = p_init[j] if t == 1 else p[j, t-1]
        v_prev = v_init[j] if t == 1 else v[j, t-1]
        m_nt.addConstr(
            p[j, t] - p_prev <= ramp_up[j] * v_prev + start_up[j] * y[j, t],
            name = f"ramp_up_limit_{j}_{t}"
        )

Constraint 2.7

$$
p_j(t) - p_j(t-1) \leq R_j^U v_j(t-1) + S_j^U y_j(t) \qquad \forall j \in J \quad \forall t \in T
$$

In [79]:
#Constraint 2.8

for j in generators:
    for t in time_periods:
        p_prev = p_init[j] if t == 1 else p[j, t-1]
        m_nt.addConstr(
            p_prev - p[j, t] <= ramp_down[j] * v[j,t] + start_down[j] * z[j, t],
            name= f"ramp_down_limit_{j}_{t}"
        )

Constraint 2.8

$$
p_j(t-1) - p_j(t) \leq R_j^D v_j(t) + S_j^D z_j(t) \qquad \forall j \in J \quad \forall t \in T
$$

In [80]:
#2.9 - Uptime constraints

total_hours = len(time_periods)

for j in generators:
    T_U = time_up[j]             # Minimum up time required (e.g., 3)
    already_on = u_init[j]      # Hours needing to be on at the start (e.g., 2)

    if v_init[j] == 1:
        remaining_needed = u_init[j]
        L_j = max(0, min(total_hours, remaining_needed))  # Unit is off, so no forced UP time, UPDATE: SWAPED THIS IN V2
    else:   
     #was remaining_needed = T_U - u_init[j] previously
        L_j = 0
    
    # The Forced Period (t = 1 to L_j)
    for t in range(1, L_j + 1):
        m_nt.addConstr(
            v[j, t] == 1,
            name=f"forced_up_{j}_{t}"
        )
    
    # The Standard Period (t = L_j + 1 to End)
    # (t - TU + 1) is guaranteed to be >= 1.
    for t in range(L_j + 1, total_hours + 1):
        
        # Calculate the start of the window
        start_window = t - T_U + 1
        start_window = max(1, start_window)
        
        # Sum startups in the window
        recent_startups = gp.quicksum(y[j, k] for k in range(start_window, t + 1))
        
        # Add the constraint
        m_nt.addConstr(
            v[j, t] >= recent_startups,
            name=f"min_up_time_{j}_{t}"
        )

COnstraint 2.9- Minimum uptime

$$
\sum_{k = t - T_j^U + 1}^t y_j(k) \leq v_j(t) \qquad \forall t \in [L_j + 1,...,|T|], \quad \forall j \in J
$$


In [81]:
# COnstraint 2.10 - minimum downtime

total_hours = len(time_periods)

for j in generators:
    T_D = time_down[j]             # Minimum down time required (e.g., 2)
    already_off = d_init[j]      # Hours already off at start (e.g., 0)

    if v_init[j] == 0:
        F_j = 0 #
    else:
        remaining_needed_off = already_off
        F_j = max(0, min(total_hours, remaining_needed_off))   

    
    # The Forced Period (t = 1 to F_j)
    for t in range(1, F_j + 1):
        m_nt.addConstr(
            v[j, t] == 0,
            name=f"forced_down_{j}_{t}"
        )
    
    # The Standard Period (t = F_j + 1 to End)
    # (t - TU + 1) is guaranteed to be >= 1.
    for t in range(F_j + 1, total_hours + 1):
        
        # Calculate the start of the window
        start_window = t - T_D + 1
        start_window = max(1, start_window)
        
        # Sum shutdowns in the window
        recent_shutdowns = gp.quicksum(z[j, k] for k in range(start_window, t + 1))
        
        # Add the constraint
        m_nt.addConstr(
            v[j, t] + recent_shutdowns <= 1,
            name=f"min_down_time_{j}_{t}"
        )

Condition 2.10 - Minimum Downtime

$$
v_j(t) + \sum_{k = t - T_j^U + 1}^t z_j(k) \leq 1 \qquad \forall t \in [F_j + 1,...,|T|], \quad \forall j \in J
$$

In [82]:
# Constraint 2.11 - Generation limit

for j in generators:
    for t in time_periods:
        m_nt.addConstr(
            p_max[j, t] <= p_max_limit[j] * v[j, t], 
            name=f"static_max_{j}_{t}"
        )
        m_nt.addConstr(
            p_min_limit[j] * v[j, t] <= p[j, t],
            name=f"min_power_limit_{j}_{t}" 
        )
        m_nt.addConstr(
            p[j, t] <= p_max[j, t],
            name=f"max_power_limit_{j}_{t}"
        )
        #m.addConstr(
            #p[j, t] <= p_max[j, t],
            #name=f"gen_le_avail_{j}_{t}"
        #)
        #m.addConstrs(
            #p[j, t] <= p_max[j, t], 
            #name=f"gen_le_avail_{j}_{t}"
        #)


Constraint 2.11 - Generating limit

$$
\underline{P_j} v_j(t) \leq p_j(t) \leq \overline{p_j}(t)\leq \overline{P_j}v_j(t) \qquad \forall j \in J, \quad \forall t \in T
$$

In [83]:
# constriant 2.12

for j in generators:
    for t in time_periods:
        p_prev = p_init[j] if t == 1 else p[j, t-1]
        v_prev = v_init[j] if t == 1 else v[j, t-1] 
        m_nt.addConstr(
            p_max[j, t] <= p_prev + ramp_up[j] * v_prev + start_up[j] * y[j, t],
            name = f"ramp_up_limit_max_{j}_{t}"
            )

Constraint 2.12 - max power outage at t

$$
\overline{p_j}(t) \leq p_j(t-1) + R_j^U v_j(t-1) + S_j^U y_j(t) \qquad \forall j \in J, \quad \forall t \in T
$$

In [84]:
# COnstraint 2.13 - limiting maximum power outage

for j in generators:
    for t in time_periods:

        z_next = z[j, t+1] if t < 6 else 0#m.addConstr(p_max[j, t] <= p_max_limit[j] * v[j, t],name=f"max_power_outage_t6_{j}_{t}")

        m_nt.addConstr(
            p_max[j, t] <= p_max_limit[j] * v[j, t] - p_max_limit[j] * z_next + z_next * start_down[j],
            name = f"max_power_outage_{j}_{t}"
        )

Constraint 2.13- power outage limit

$$
\overline{p_j}(t) \leq \overline{P_j}\left[ v_j(t) - z_j(t + 1) \right] + z_j(t+1) S_j^D \qquad \forall j \in J, \quad \forall t \in T
$$

In [85]:
# Optimisation

m_nt.optimize()



Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 12th Gen Intel(R) Core(TM) i5-1250P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Optimize a model with 324 rows, 126 columns and 893 nonzeros (Min)
Model fingerprint: 0xb5ab8b89
Model has 36 linear objective coefficients
Variable types: 72 continuous, 54 integer (54 binary)
Coefficient statistics:
  Matrix range     [8e-01, 3e+02]
  Objective range  [5e+00, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+02]

Presolve removed 217 rows and 73 columns
Presolve time: 0.00s
Presolved: 107 rows, 53 columns, 356 nonzeros
Variable types: 30 continuous, 23 integer (23 binary)
Found heuristic solution: objective 14200.000000
Found heuristic solution: objective 12950.000000

Root relaxation: objective 9.629979e+03, 39 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bo

In [86]:
# Dual Variables too hard to extract, thus we can just see by marginal cost of units what duals is. 

In [87]:
print("\n" + "="*80)
print("CHAPTER 3.4.1 - NETWORK-CONSTRAINED UC RESULTS")
print("="*80)

# 1. Optimal Objective Value
print(f"\nTotal Cost: £{m_nt.ObjVal:.2f}")

# 2. Generation Schedule (Matches Document Table, Page 242)
print("\n" + "-"*80)
print("GENERATION SCHEDULE (MW)")
print("-"*80)
print(f"{'Time':<6}", end="")
for j in generators:
    print(f"{'Gen ' + str(j):<10}", end="")
print()
print("-"*80)

for t in time_periods:
    print(f"{t:<6}", end="")
    for j in generators:
        p_val = p[j, t].X
        print(f"{p_val:<10.2f}", end="")
    print()

# 3. Nodal Energy Prices (Dual Variables - Page 242)
print("\n" + "-"*80)
print("NODAL ENERGY PRICES (£/MWh) - Dual of Nodal Balance Constraint")
print("-"*80)
print(f"{'Time':<6}", end="")
for n in nodes:
    print(f"{'Node ' + str(n):<10}", end="")
print()
print("-"*80)

#for t in time_periods:
    #print(f"{t:<6}", end="")
    #for n in nodes:
        # Get dual value from nodal balance constraint
        #dual_val = m_nt.getAttr('Pi', m_nt.getConstrByName(f"Nodal_Balance[{n},{t}]"))
        #print(f"{dual_val:<10.2f}", end="")
    #print()

# 4. Unit Commitment Status (Page 243)
print("\n" + "-"*80)
print("UNIT COMMITMENT STATUS")
print("-"*80)
print(f"{'Time':<6}", end="")
for j in generators:
    print(f"{'Unit ' + str(j):<10}", end="")
print()
print("-"*80)

for t in time_periods:
    print(f"{t:<6}", end="")
    for j in generators:
        v_val = int(v[j, t].X)
        status = "on" if v_val == 1 else "off"
        print(f"{status:<10}", end="")
    print()

# 5. Transmission Line Flows (Page 243) - Example for t=1
print("\n" + "-"*80)
print("POWER FLOWS ON TRANSMISSION LINES (MW) - Time Period 1")
print("-"*80)
print(f"{'From':<6} | {'To':<6} | {'Flow (MW)':<12} | {'Capacity':<10} | {'Utilisation':<12}")
print("-"*80)

t_example = 1
for (n, m) in lines:
    flow = B[(n, m)] * (theta[n, t_example].X - theta[m, t_example].X)
    capacity = P_line[(n, m)]
    utilisation = (abs(flow) / capacity) * 100 if capacity > 0 else 0
    print(f"{n:<6} | {m:<6} | {flow:<12.2f} | {capacity:<10} | {utilisation:<12.2f}%")

# 6. Reserve Information
print("\n" + "-"*80)
print("SPINNING RESERVE AVAILABLE (MW)")
print("-"*80)
for t in time_periods:
    total_capacity = sum(p_max_limit[j] * v[j, t].X for j in generators)
    total_generation = sum(p[j, t].X for j in generators)
    reserve_available = total_capacity - total_generation
    reserve_required = sum(nodal_demand[(n, t)] for n in nodes) + reserve[t] - total_generation
    print(f"Time {t}: Available = {reserve_available:.2f} MW, Required = {reserve[t]} MW")

print("\n" + "="*80)


CHAPTER 3.4.1 - NETWORK-CONSTRAINED UC RESULTS

Total Cost: £9900.00

--------------------------------------------------------------------------------
GENERATION SCHEDULE (MW)
--------------------------------------------------------------------------------
Time  Gen 1     Gen 2     Gen 3     
--------------------------------------------------------------------------------
1     160.00    50.00     30.00     
2     200.00    50.00     0.00      
3     200.00    0.00      0.00      
4     170.00    0.00      0.00      
5     200.00    0.00      30.00     
6     190.00    0.00      0.00      

--------------------------------------------------------------------------------
NODAL ENERGY PRICES (£/MWh) - Dual of Nodal Balance Constraint
--------------------------------------------------------------------------------
Time  Node 1    Node 2    Node 3    Node 4    Node 5    Node 6    
--------------------------------------------------------------------------------

---------------------------

In [90]:
if m_nt.status == GRB.INFEASIBLE:
    print("Model is Infeasible. Computing IIS to find the conflict...")
    m.computeIIS()
    m.write("conflict.ilp")
    print("Conflict saved to 'conflict.ilp'. Open this file to see the bad constraints.")
    
    # Optional: Print the conflicting constraints directly to the screen
    for c in m.getConstrs():
        if c.IISConstr:
            print(f"Conflict found in: {c.ConstrName}")


In [91]:
#!jupyter nbconvert --to pdf Basic_model_V2.ipynb